# Node 1 — EDA: MovieLens 32M

Purpose: not general exploration — every chart here informs a specific design decision in later nodes.

## Dataset provenance (from GroupLens README)

> This dataset (ml-32m) describes 5-star rating and free-text tagging activity from MovieLens.
> It contains 32,000,204 ratings and 2,000,072 tag applications across 87,585 movies.
> These data were created by 200,948 users between January 09, 1995 and October 12, 2023.
> Users were selected at random for inclusion. **All selected users had rated at least 20 movies.**
> No demographic information is included.

**Implications we're designing around:**
- Item-side cold-start is real and observable (movies can have 0 ratings). User-side cold-start is
  **structurally hidden** by the ≥20-rating inclusion criterion — every user here already has history.
  This is a known limitation of using MovieLens as a proxy; node 5's simulation can still *create*
  new users mid-stream if we want to exercise that path.
- No demographic fields exist → user representations must be behavioral/session-based, not
  demographic-based, by necessity of the data, not just design preference.
- Confirmed timestamp range: 1995-01-09 → 2023-10-12. This is the window the event-stream
  generator (node 1, later script) will replay.

## Schema (confirmed from raw CSVs)
- `ratings.csv`: userId, movieId, rating, timestamp (unix seconds)
- `movies.csv`: movieId, title (includes release year in parens), genres (pipe-separated)
- `links.csv`: movieId, imdbId, tmdbId
- `tags.csv`: userId, movieId, tag, timestamp

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DATA_DIR = Path("../data/raw/ml-32m")  # adjust if your notebook lives elsewhere
CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 150

ratings_lazy = pl.scan_csv(DATA_DIR / "ratings.csv")
movies_lazy = pl.scan_csv(DATA_DIR / "movies.csv")

print(ratings_lazy.collect_schema())
print(movies_lazy.collect_schema())

## 0. Extract release year from movie title

Titles look like `Toy Story (1995)`. We need this as its own column for:
- sanity-checking rating timestamps against release date (a rating that predates the release is a
  data-quality flag worth knowing about),
- a potential `movie_age_at_rating_time` feature later in the pipeline.

In [ ]:
movies = (
    movies_lazy
    .with_columns(
        pl.col("title").str.extract(r"\((\d{4})\)\s*$", 1).cast(pl.Int32).alias("release_year")
    )
    .collect()
)

n_missing_year = movies["release_year"].is_null().sum()
print(f"Movies without a parseable release year: {n_missing_year} / {movies.height}")
movies.filter(pl.col("release_year").is_null()).head(5)  # eyeball what didn't match

## 1. Ratings volume over time
Confirms density/gaps in the timestamp range before building the event-stream generator on top of it.

In [ ]:
daily_counts = (
    ratings_lazy
    .with_columns(pl.from_epoch("timestamp", time_unit="s").dt.date().alias("date"))
    .group_by("date")
    .agg(pl.len().alias("n_ratings"))
    .sort("date")
    .collect()
)

print(f"Date range: {daily_counts['date'].min()} → {daily_counts['date'].max()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_counts["date"], daily_counts["n_ratings"], linewidth=0.6)
ax.set_title("Ratings volume per day")
ax.set_xlabel("Date")
ax.set_ylabel("Number of ratings")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
fig.tight_layout()
fig.savefig(CHARTS_DIR / "ratings_volume_over_time.png", bbox_inches="tight")
plt.show()

## 2. Global cold-start check: movies with zero ratings

`movies.csv` and `ratings.csv` are not the same population — a movie can exist in the catalog with
**no ratings at all**. Anti-join catches what a groupby on `ratings` alone would silently miss.

In [ ]:
rated_movie_ids = ratings_lazy.select("movieId").unique()

zero_rating_movies = (
    movies.lazy()
    .join(rated_movie_ids, on="movieId", how="anti")
    .collect()
)

n_total_movies = movies.height
n_zero = zero_rating_movies.height
print(f"Total movies: {n_total_movies:,}")
print(f"Movies with 0 ratings (all-time, global snapshot): {n_zero:,} ({n_zero / n_total_movies * 100:.1f}%)")
print("\nNote: this is a STATIC, all-time number. See section 4 for the time-aware version,")
print("which is the one that actually matters for a leakage-safe pipeline.")
zero_rating_movies.select("title", "genres", "release_year").sample(10)

## 3. Ratings per movie / per user — long tail (log-log, correctly binned)

Bug to avoid: `plt.hist(..., bins=50, log=True)` uses **linearly-spaced** bins even when the axis is
displayed on a log scale — nearly everything falls into the first bin. Bins must be generated with
`np.logspace` to match a log-scaled axis.

In [ ]:
ratings_per_movie = ratings_lazy.group_by("movieId").agg(pl.len().alias("n_ratings")).collect()

median_r = ratings_per_movie["n_ratings"].median()
pct_under_10 = (ratings_per_movie["n_ratings"] < 10).mean() * 100
print(f"Median ratings/movie (movies with ≥1 rating only): {median_r:.0f}")
print(f"Of movies with ≥1 rating, {pct_under_10:.1f}% have < 10 ratings")
print(f"Plus the {n_zero:,} zero-rating movies from section 2, not included in this groupby at all.")

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(0, np.log10(ratings_per_movie["n_ratings"].max()), 50)
ax.hist(ratings_per_movie["n_ratings"], bins=bins)
ax.set_xscale("log")
ax.set_yscale("log")
ax.axvline(1, color="red", linestyle="--", linewidth=1)
ax.text(1.1, ax.get_ylim()[1] * 0.7,
        f"  +{n_zero:,} movies have\n  0 ratings\n  (not plottable on log axis)",
        color="red", fontsize=8, va="top")
ax.set_title("Ratings per movie (long tail)")
ax.set_xlabel("Number of ratings (log scale)")
ax.set_ylabel("Number of movies (log scale)")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "ratings_per_movie.png", bbox_inches="tight")
plt.show()

In [ ]:
ratings_per_user = ratings_lazy.group_by("userId").agg(pl.len().alias("n_ratings")).collect()

min_u = ratings_per_user["n_ratings"].min()
median_u = ratings_per_user["n_ratings"].median()
print(f"Min ratings/user: {min_u}  (expect ≥20 per the dataset README — confirms no user-side cold-start exists here)")
print(f"Median ratings/user: {median_u:.0f}")

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(np.log10(min_u), np.log10(ratings_per_user["n_ratings"].max()), 50)
ax.hist(ratings_per_user["n_ratings"], bins=bins)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Ratings per user (long tail) — note: floor is 20 by dataset construction")
ax.set_xlabel("Number of ratings (log scale)")
ax.set_ylabel("Number of users (log scale)")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "ratings_per_user.png", bbox_inches="tight")
plt.show()

## 4. Time-aware cold-start (the one that actually matters)

Section 2's 3.6% figure is a **global, all-time** snapshot — "as of the 2023 export, which movies
have zero ratings ever." That's not what the feature pipeline needs. A movie released in 1995 with
50,000 ratings *today* had **zero** ratings on its release day. Cold-start is a property of
**(movie, point in time)**, not a static per-movie label.

This computes, for every rating event, how many prior ratings that movie had *at that point* —
leakage-safe by construction, since it only ever looks backward from each event's own timestamp.
The threshold below (`COLD_START_THRESHOLD`) is a placeholder parameter, not a final decision —
revisit when node 3's ranking model is being designed.

In [ ]:
COLD_START_THRESHOLD = 10  # fewer than this many PRIOR ratings = cold-start at that point in time

ratings_sorted = ratings_lazy.sort("timestamp")

movie_cumcount = (
    ratings_sorted
    .with_columns(
        pl.int_range(1, pl.len() + 1).over("movieId").alias("nth_rating_for_movie")
    )
    .with_columns(
        (pl.col("nth_rating_for_movie") <= COLD_START_THRESHOLD).alias("is_cold_start_at_this_point"),
        pl.from_epoch("timestamp", time_unit="s").dt.year().alias("rating_year"),
    )
    .collect()
)

cold_start_rate_by_year = (
    movie_cumcount
    .group_by("rating_year")
    .agg(pl.col("is_cold_start_at_this_point").mean().alias("pct_cold_start_events"))
    .sort("rating_year")
)

overall_pct = movie_cumcount["is_cold_start_at_this_point"].mean() * 100
print(f"Across ALL rating events (not just distinct movies), {overall_pct:.1f}% happened while")
print(f"the movie had fewer than {COLD_START_THRESHOLD} prior ratings.")
print("This is the number that reflects what the model actually sees during training/serving —")
print("compare it to section 2's static 3.6%, which only counts distinct never-rated movies.")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cold_start_rate_by_year["rating_year"], cold_start_rate_by_year["pct_cold_start_events"] * 100, marker="o")
ax.set_title(f"% of rating events where movie had < {COLD_START_THRESHOLD} prior ratings, by year")
ax.set_xlabel("Year")
ax.set_ylabel("% of rating events (cold-start)")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "cold_start_rate_over_time.png", bbox_inches="tight")
plt.show()

## 5. Movie age at rating time (data-quality sanity check)

Using the extracted `release_year`, compute how many years after release each rating occurred.
A negative value (rated before release) is a data-quality flag worth knowing about — either a
title-parsing error (wrong year extracted) or a genuine metadata quirk (e.g. re-releases, festival
screenings predating wide release).

In [ ]:
ratings_with_year = (
    ratings_lazy
    .with_columns(pl.from_epoch("timestamp", time_unit="s").dt.year().alias("rating_year"))
    .join(movies.lazy().select("movieId", "release_year"), on="movieId", how="left")
    .with_columns((pl.col("rating_year") - pl.col("release_year")).alias("years_since_release"))
    .collect()
)

n_negative = (ratings_with_year["years_since_release"] < 0).sum()
pct_negative = n_negative / ratings_with_year.height * 100
print(f"Ratings that occurred BEFORE the parsed release year: {n_negative:,} ({pct_negative:.2f}%)")
print("If this is near 0%, release_year extraction is trustworthy for a movie_age feature.")
print("If it's notably high, investigate: likely a subset of titles with unusual parenthetical formatting.")

fig, ax = plt.subplots(figsize=(8, 4))
clipped = ratings_with_year["years_since_release"].clip(0, 50)  # clip for a readable view
ax.hist(clipped, bins=50)
ax.set_title("Years between movie release and rating (clipped 0–50)")
ax.set_xlabel("Years since release")
ax.set_ylabel("Number of ratings")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "years_since_release.png", bbox_inches="tight")
plt.show()

## Takeaways for the README (fill in the printed numbers after running)

- Dataset spans 1995-01-09 → 2023-10-12; volume shape: (describe gaps/bursts from section 1).
- Static cold-start (section 2): __% of movies have zero ratings, ever.
- Time-aware cold-start (section 4): __% of *rating events* occur while the movie had fewer than
  {COLD_START_THRESHOLD} prior ratings — this is the number that drives node 3's design, and it's
  higher than the static one because it counts every early rating, not just permanently-unrated movies.
- User-side cold-start does not exist in this dataset (min ratings/user = __, expected ≥20) —
  documented limitation, addressed synthetically in node 5 if needed.
- Release-year extraction failed for __ movies and produced __ negative-age ratings —
  (trustworthy / needs a fallback for those edge cases).
- These four PNGs get embedded directly in `01_feature_pipeline/README.md`; this notebook is the
  full record, linked from there.